# SGR: Synthetic Spectra Demonstration

This notebook uses the **CPU implementation** with synthetic spectra. It is not a reproduction of the thesis asteroid results or an independent physical validation.

Install `requirements.txt` and run the cells. The synthetic split is accepted only if the BIC and multi-view consensus criteria are satisfied.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
root = Path.cwd()
if not (root / "spectral_geometry_refinement_cpu.py").exists():
    root = root.parent
sys.path.insert(0, str(root))
from spectral_geometry_refinement_cpu import SGRConfig, run_sgr
rng = np.random.default_rng(42)
wave = np.linspace(0.45, 2.45, 81)
population = np.repeat([0, 1], 55)
centers = np.where(population == 0, 0.94, 1.10)
X = np.array([1 + 0.04 * (wave - 0.55) - 0.30 * np.exp(-0.5 * ((wave - center) / 0.10)**2) - 0.10 * np.exp(-0.5 * ((wave - 1.95) / 0.16)**2) for center in centers])
X += rng.normal(0, 0.004, X.shape)
labels = np.zeros(len(X), dtype=int)
for k in (0,1):
    plt.plot(wave, X[population == k].mean(axis=0), label=f"Synthetic population {k+1}")
plt.xlabel("Wavelength (µm)")
plt.ylabel("Relative reflectance")
plt.legend()
plt.show()

## Run SGR

The demonstration reduces `gmm_n_init` for speed. This configuration is **not** the reference thesis setting.

In [ ]:
cfg = SGRConfig(max_refine_depth=1, min_parent_n=40, gmm_n_init=2)
result = run_sgr(X, wave, labels, config=cfg, make_plots=False)
print(result.history.to_string(index=False))
print("Original groups:", result.n_global_clusters)
print("Refined groups:", result.n_refined_clusters)
print("Accepted splits:", result.n_splits)

## Deterministic stopping-rule check

This checks the minimum-parent-size condition, not a statistical control-negative experiment.

In [ ]:
stop = run_sgr(X[:24], wave, np.repeat([0,1],12), config=SGRConfig(min_parent_n=80), make_plots=False)
assert stop.n_splits == 0
assert stop.n_refined_clusters == 2